# Campus Eco & Waste Sorting Identifier: Training Notebook (TrashNet)

Fine-tunes **MobileNetV3-Large** (ImageNet weights) on the **TrashNet** dataset's 6 classes:
`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.

The website then groups these into bins: cardboard + paper → Paper/Cardboard, plastic → Plastics,
glass → Glass, metal → Metal, trash → Landfill (see `backend/data/classes.json`).

**Before running:**
1. **Settings** → Accelerator: **GPU T4**, Internet: **On**.
2. TrashNet: attach it with **Add Input** if you like, but you don't have to. If none is attached, the
   notebook downloads the original TrashNet (43 MB) from GitHub automatically.

Then click **Run All** (about 15–20 minutes on a T4). At the end, download `mobilenetv3_waste.pth` from the
**Output** tab and put it in `campus-eco-sorter/backend/model/` on your laptop.

## 1. Imports and settings

In [ ]:
import os, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

# The website expects EXACTLY this order (backend/data/classes.json). Don't change it.
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Leave as None to find TrashNet automatically. If that fails: in the right panel hover over
# 'dataset-resized', click the copy-path icon, and paste it here, e.g. DATA_DIR = "/kaggle/input/trashnet/dataset-resized"
DATA_DIR = None

# Backup: download TrashNet directly (needs Internet ON). Open the dataset's Kaggle page and copy the
# part of the URL after kaggle.com/datasets/  e.g. for kaggle.com/datasets/someone/trashnet -> "someone/trashnet"
KAGGLE_DATASET = None

# Final backup, used automatically if nothing else works: the original TrashNet zip on GitHub
TRASHNET_URL = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"

BATCH_SIZE = 32
HEAD_EPOCHS = 5       # phase 1: train only the new final layer
FINETUNE_EPOCHS = 10   # phase 2: train the whole network gently
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

## 2. Find TrashNet
Kaggle puts inputs under `/kaggle/input/`, and the folder layout depends on which TrashNet copy you added.
This cell finds the 6 class folders automatically. **Check the counts:** TrashNet has about 2,500 images
(roughly 400–600 per class, but only ~140 for trash).

In [ ]:
import re, zipfile, tarfile

INPUT = Path("/kaggle/input")
EXTRACT = Path("/tmp/trashnet_extracted")   # outside /kaggle/working, so it stays out of Output
ARCHIVE_EXTS = (".zip", ".tar", ".tar.gz", ".tgz")

SKIP = {"lib", "proc", "sys", "site-packages", "dist-packages", ".git", "__MACOSX"}

def walk(root):
    # followlinks=True: Kaggle often attaches datasets as linked folders
    seen = set()
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(dirpath)
        if real in seen:          # avoid looping through circular links
            dirnames[:] = []
            continue
        seen.add(real)
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP)
        yield Path(dirpath), dirnames, sorted(filenames)

def images_in(folder):
    return [d / f for d, _, files in walk(folder) for f in files
            if Path(f).suffix.lower() in IMG_EXTS and not f.startswith("._")]

def find_class_dirs(roots):
    # Some TrashNet uploads contain every image twice (e.g. two copies of 'dataset-resized').
    # Keep only ONE folder per class so the same photo can't end up in both train and test.
    found = {}
    for root in roots:
        for d, dirnames, _ in walk(root):
            for sub in dirnames:
                name = sub.lower()
                if name in CLASS_TO_IDX and name not in found and images_in(d / sub):
                    found[name] = d / sub
    return found

def show_tree(root, max_depth=4):
    root = Path(root)
    for d, dirnames, files in walk(root):
        depth = len(d.relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        print("    " * depth + f"{d.name}/  ({len(files)} files)", ", ".join(files[:3]))

# Where Kaggle may have mounted the data (it differs between Kaggle versions)
CANDIDATES = [INPUT, Path("/kaggle"), Path("/mnt"), Path.home() / ".cache" / "kagglehub", Path("/datasets")]
if DATA_DIR:
    roots = [Path(DATA_DIR)]
    assert roots[0].exists(), f"DATA_DIR {DATA_DIR} does not exist - re-copy the path from the Input panel."
else:
    roots = [p for p in CANDIDATES if p.exists()]
class_dirs = find_class_dirs(roots)

# Backup 1: download a Kaggle copy by its ID (only if you set KAGGLE_DATASET)
if len(class_dirs) < len(CLASSES) and KAGGLE_DATASET:
    try:
        import kagglehub
        print("Downloading", KAGGLE_DATASET, "from Kaggle ...")
        roots.append(Path(kagglehub.dataset_download(KAGGLE_DATASET)))
        class_dirs = find_class_dirs(roots)
    except Exception as e:
        print("  Kaggle download failed:", str(e)[:200])

# Backup 2: download the ORIGINAL TrashNet zip from GitHub (43 MB, needs Internet ON).
# This always works, even when no dataset is attached.
if len(class_dirs) < len(CLASSES):
    import urllib.request
    zip_path = Path("/tmp/dataset-resized.zip")
    if not zip_path.exists():
        print("Downloading TrashNet from GitHub ...")
        urllib.request.urlretrieve(TRASHNET_URL, zip_path)
    out = EXTRACT / "github"
    if not out.exists():
        zipfile.ZipFile(zip_path).extractall(out)
    roots.append(out)
    class_dirs = find_class_dirs(roots)

print("Searching in:", [str(r) for r in roots])

# 1) Some uploads are a .zip (sometimes a zip inside a zip) - extract them
for _ in range(2):
    if len(class_dirs) == len(CLASSES):
        break
    archives = [d / f for r in roots for d, _, files in walk(r) for f in files if f.lower().endswith(ARCHIVE_EXTS)]
    for a in archives:
        out = EXTRACT / a.name.split(".")[0]
        if out.exists():
            continue
        print("Extracting", a)
        if a.name.lower().endswith(".zip"):
            zipfile.ZipFile(a).extractall(out)
        else:
            tarfile.open(a).extractall(out)
    if archives and EXTRACT not in roots:
        roots.append(EXTRACT)
    class_dirs = find_class_dirs(roots)

files_by_class = {c: images_in(class_dirs[c]) for c in class_dirs}

# 2) Some uploads put all images in one folder, labelled only by file name (cardboard1.jpg, glass12.jpg ...)
if len(class_dirs) < len(CLASSES):
    seen = set()
    for root in roots:
        for img in images_in(root):
            m = re.match(r"[a-z]+", img.stem.lower())
            label = m.group(0) if m else None
            if label in CLASS_TO_IDX and label not in class_dirs and img.name.lower() not in seen:
                seen.add(img.name.lower())
                files_by_class.setdefault(label, []).append(img)

missing = [c for c in CLASSES if not files_by_class.get(c)]
if missing:
    print("Couldn't find TrashNet. Here is what the searched folders contain - paste this if you need help:\n")
    for r in roots:
        show_tree(r, max_depth=3)
    raise RuntimeError(
        f"Couldn't find images for {missing}. In the Input panel, hover over 'dataset-resized', "
        "copy its path, and set DATA_DIR in the first code cell - or set KAGGLE_DATASET to download it."
    )

samples = []
for c in CLASSES:
    files = sorted(files_by_class[c])
    samples += [(str(p), CLASS_TO_IDX[c]) for p in files]
    src = class_dirs.get(c, "file names")
    print(f"{c:10s} {len(files):5d} images   from {src}")
print("Total:", len(samples))

## 3. Split into train / validation / test
70% train, 15% validation (used to pick the best epoch), 15% test (only touched once, at the end).
`stratify` keeps the same class proportions in every split.

In [ ]:
labels = [l for _, l in samples]
train_samples, rest = train_test_split(samples, test_size=0.30, stratify=labels, random_state=SEED)
val_samples, test_samples = train_test_split(rest, test_size=0.50, stratify=[l for _, l in rest], random_state=SEED)
print(f"train {len(train_samples)}, val {len(val_samples)}, test {len(test_samples)}")

## 4. Image transforms and data loaders

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# TrashNet photos are all on a plain white background. Strong augmentation helps the model
# cope with real phone photos (different angles, lighting, backgrounds, blur).
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# MUST match build_transform() in backend/model.py
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class WasteDataset(Dataset):
    def __init__(self, samples, tf):
        self.samples, self.tf = samples, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        return self.tf(img), label

loader_args = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(DEVICE == "cuda"))
train_loader = DataLoader(WasteDataset(train_samples, train_tf), shuffle=True, **loader_args)
val_loader = DataLoader(WasteDataset(val_samples, eval_tf), shuffle=False, **loader_args)

## 5. Build the model
MobileNetV3-Large pre-trained on ImageNet, with the last layer replaced by a 6-class layer.
Class weights give the small `trash` class more importance during training.

In [ ]:
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(CLASSES))   # same as backend/model.py
model = model.to(DEVICE)

counts = Counter(l for _, l in train_samples)
weights = torch.tensor([len(train_samples) / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                       dtype=torch.float, device=DEVICE)
print("Class weights:", {c: round(w, 2) for c, w in zip(CLASSES, weights.tolist())})
criterion = nn.CrossEntropyLoss(weight=weights)

## 6. Training loop

In [ ]:
SAVE_PATH = "/kaggle/working/mobilenetv3_waste.pth"
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_acc = 0.0

def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def train(epochs, optimizer, scheduler=None, phase=""):
    global best_acc
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(train_loader, optimizer)
        va_loss, va_acc = run_epoch(val_loader)
        if scheduler: scheduler.step()
        history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)
        saved = ""
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), SAVE_PATH)
            saved = "  <- best, saved"
        print(f"[{phase} {ep}/{epochs}] train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time()-t0:.0f}s{saved}")

## 7. Phase 1: train only the new last layer
The pre-trained layers are frozen, so the new layer learns quickly without disturbing them.

In [ ]:
for p in model.features.parameters():
    p.requires_grad = False
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
train(HEAD_EPOCHS, opt, phase="head")

## 8. Phase 2: fine-tune the whole network
Unfreeze everything and train with a small learning rate that slowly decreases.

In [ ]:
for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS)
train(FINETUNE_EPOCHS, opt, sched, phase="finetune")
print(f"Best validation accuracy: {best_acc:.3f}")

In [ ]:
# Training curves (useful for the report)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["val_acc"]); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch")
ax[1].axvline(HEAD_EPOCHS - 0.5, ls="--", c="gray")
plt.tight_layout(); plt.savefig("/kaggle/working/training_curves.png", dpi=150); plt.show()

## 9. Evaluate on the test set (Week 1 task)
Loads the best saved model and reports accuracy, per-class precision / recall / F1 and a confusion matrix.
It also reports accuracy after grouping into the **website's bins** (cardboard and paper count as the same bin),
which is the number that matters to users.

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

test_loader = DataLoader(WasteDataset(test_samples, eval_tf), shuffle=False, **loader_args)
y_true, y_pred, y_conf = [], [], []
t0 = time.time()
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x.to(DEVICE)), 1)
        conf, pred = probs.max(1)
        y_pred += pred.cpu().tolist(); y_conf += conf.cpu().tolist(); y_true += y.tolist()

y_true, y_pred, y_conf = np.array(y_true), np.array(y_pred), np.array(y_conf)
print(f"Test images: {len(y_true)}")
print(f"Accuracy (6 TrashNet classes): {np.mean(y_true == y_pred):.3f}")

# Same grouping as backend/data/classes.json -> label_to_category
TO_BIN = {"cardboard": "paper", "glass": "glass", "metal": "metal", "paper": "paper", "plastic": "plastics", "trash": "landfill"}
bin_true = np.array([TO_BIN[CLASSES[i]] for i in y_true]); bin_pred = np.array([TO_BIN[CLASSES[i]] for i in y_pred])
print(f"Accuracy (website bins):       {np.mean(bin_true == bin_pred):.3f}")
print(f"Predictions below 60% confidence: {np.mean(y_conf < 0.60):.1%}  (the site asks the user to check these)\n")

print(classification_report(y_true, y_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Confusion matrix - TrashNet test set"); plt.xticks(rotation=30); plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Look at some mistakes - useful for the error-analysis part of the report
wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} misclassified test images")
show = wrong[:8]
if len(show):
    fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 3))
    for ax, i in zip(np.atleast_1d(axes), show):
        ax.imshow(Image.open(test_samples[i][0]).convert("RGB")); ax.axis("off")
        ax.set_title(f"true: {CLASSES[y_true[i]]}\npred: {CLASSES[y_pred[i]]} ({y_conf[i]:.0%})", fontsize=9)
    plt.tight_layout(); plt.savefig("/kaggle/working/mistakes.png", dpi=120); plt.show()

## 10. Try your own photo
Upload a phone photo with **Add Input → Upload** (or drag it into the file panel), set its path below, and run.
TrashNet only has plain white backgrounds, so this shows how well the model handles real-world photos.

In [ ]:
def predict(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(model(eval_tf(img).unsqueeze(0).to(DEVICE)), 1)[0].cpu()
    top = probs.argsort(descending=True)[:3]
    plt.imshow(img); plt.axis("off")
    plt.title(", ".join(f"{CLASSES[i]} {probs[i]:.0%}" for i in top)); plt.show()

MY_PHOTO = None   # e.g. "/kaggle/input/my-photos/bottle.jpg"
predict(MY_PHOTO or random.choice(test_samples)[0])

## 11. Done: download the model
Open the **Output** tab (right panel, or after *Save Version*) and download:
- `mobilenetv3_waste.pth` → put in `campus-eco-sorter/backend/model/`
- `training_curves.png`, `confusion_matrix.png`, `mistakes.png` → for your report

`CLASSES` above already matches the website's `classes.json`, so no other changes are needed.

In [ ]:
print("Class order:", CLASS_TO_IDX)
print("Saved model:", SAVE_PATH, f"({os.path.getsize(SAVE_PATH)/1e6:.1f} MB)")